# Phase D — GRPO training

Trains the adaptive policy with TRL `GRPOTrainer` + LoRA. **Runtime → Change runtime type → GPU.**

### Run order (from the proposal)

1. **Smoke test** — 5 steps, tiny group. Answers only "does it execute".
2. **Go/no-go gate** — `--lambda-think 0.0`, i.e. correctness + format, no length penalty.
   If GRPO cannot improve plain correctness over base, the adaptive question is moot and the
   project pivots to characterising why.
3. **λ sweep** — only after the gate passes.

### What to watch

`metric_think_rate` in the logs, not the reward curve. Collapse to all-think or all-no-think
is the primary failure mode and it is invisible in reward alone — a policy that stopped
reasoning and one that started emitting malformed calls both flatten it.

From the fp16 baselines: the **oracle thinks on 17.4%** of items, prompting alone gives 96.9%.
A trained policy near 17–20% is in the right regime; near 0% or near 100% is collapse.

### Why λ is swept where it is

Break-even λ per category, computed from the fp16 baselines: `simple_python` never worth
thinking (Δ = −2.8%), `multiple` 0.07, `parallel` 0.48, `parallel_multiple` 0.53,
`irrelevance` 2.11. Four of five switch off below 0.55, then nothing changes until 2.11 —
so `{0.05, 0.1, 0.25, 0.5, 1.0, 2.0}` covers every transition and a linear sweep would not.

## 1 — Install

Four steps, and **the order matters** — each one is a lesson from `notes/engineering_log.md`:

1. **vLLM** (entry 9). Rollout generation, not the optimizer, is what makes a GRPO step
   expensive. With HF `generate` a step at 16 completions x 768 tokens takes ~7 min, so 200
   steps is ~24 h — it does not fit a Colab session. vLLM's continuous batching is the same
   ~20x that turned a 16 h baseline run into 36 min.
2. **trl / peft / datasets.**
3. **torch from the cu130 index** (entry 10). vLLM 0.26 ships a CUDA 13 binary; Colab preinstalls
   `torch 2.11.0+cu128`. pip sees the version `2.11.0` as satisfying vLLM's pin and leaves torch
   alone, so a cu13 `.so` hunts for `libcudart.so.13` on a cu12 system. Reinstalling the *same
   version* from the cu130 index fixes it. This must come **after** vLLM, or vLLM's install
   silently reverts it.
4. **Uninstall torchao** (entry 13). Nothing uses it, but PEFT's LoRA dispatcher calls
   `is_torchao_available()`, which *raises* on Colab's 0.10.0 instead of returning False.

`bfcl-eval` stays `--no-deps` (entries 7 and 14): training runs the checker once per rollout, but
`scoring.bfcl_scorer` supplies the one boolean its 81-package import chain exists to resolve.

**This install takes ~10 min and will almost certainly ask to RESTART SESSION. Take it, then
resume at step 2.**

In [ ]:
# 1. vLLM first — it has the strongest opinions about torch.
!pip install -q vllm

# 2. Training stack.
!pip install -q trl peft datasets accelerate

# 3. Repair the CUDA ABI. Same torch VERSION, cu130 build. Must run after vLLM.
import torch
V = torch.__version__.split("+")[0]
print("torch before:", torch.__version__)
!pip install -q --force-reinstall torch=={V} torchvision torchaudio --index-url https://download.pytorch.org/whl/cu130

# 4. PEFT's LoRA dispatcher raises on Colab's old torchao instead of skipping it.
!pip uninstall -y -q torchao

# 5. BFCL data + checker, without its 81-package handler chain.
!pip install -q --no-deps bfcl-eval==2026.3.23

print("\n>>> If Colab offers RESTART SESSION, take it, then continue at step 2. <<<")

## 2 — Verify, then get the code

In [ ]:
import importlib.util
import os

import bfcl_eval
import peft
import torch
import trl
import vllm

data_dir = os.path.join(os.path.dirname(bfcl_eval.__file__), "data")
print("torch     :", torch.__version__, "| CUDA", torch.version.cuda)
print("vllm      :", vllm.__version__)
print("trl       :", trl.__version__)
print("peft      :", peft.__version__)
print("bfcl data :", os.path.isdir(data_dir))
print("bf16      :", torch.cuda.is_bf16_supported(), "(False on T4 — fp16 fallback)")

# PEFT's LoRA dispatcher raises rather than skipping when an old torchao is
# present, so its absence is a precondition, not a preference.
print("torchao   :", "absent (good)" if importlib.util.find_spec("torchao") is None else "PRESENT — rerun the uninstall")

assert os.path.isdir(data_dir), "BFCL data missing — rerun the --no-deps install"
assert importlib.util.find_spec("torchao") is None, "torchao still installed; PEFT will raise on LoRA injection"
# `import vllm` succeeding is the real test of the cu130 repair — it is where the
# libcudart.so.13 ImportError surfaces if torch is still a cu12 build.
print("\nenvironment OK")
!nvidia-smi --query-gpu=name,memory.total,compute_cap --format=csv

In [ ]:
REPO = "https://github.com/widodu77/rs_aidams.git"

if os.path.isdir("/content/rs_aidams"):
    !cd /content/rs_aidams && git pull --ff-only
else:
    !git clone -q $REPO /content/rs_aidams

%cd /content/rs_aidams
os.environ["PYTHONPATH"] = "src"
!git log --oneline -1

In [ ]:
from google.colab import drive

drive.mount("/content/drive")
DRIVE_RUNS = "/content/drive/MyDrive/rs_aidams/runs"
os.makedirs(DRIVE_RUNS, exist_ok=True)

## 3 — Smoke test

Five steps at the real completion length (768), through vLLM. Two things to read off:

- **`step_time`** — this is the number that decides whether the sweep is feasible at all.
  Multiply by 200 for a full run, then by 7 for gate + six λ values. Without vLLM it was
  ~37 s/step at only 4 completions x 256 tokens, which extrapolates to ~24 h for one run.
- **`frac_reward_zero_std`** — the fraction of groups where every rollout scored identically.
  Those steps produce zero advantage and therefore zero gradient. The first smoke run had this
  at 1.0 on three of five steps at group size 4; group size 8 should reduce it.

Read nothing scientific into five steps.

In [ ]:
!python -m train.train_grpo \
    --output runs/smoke \
    --lambda-think 0.0 \
    --use-vllm \
    --max-steps 5 \
    --num-generations 8 \
    --per-device-batch-size 8 \
    --gradient-accumulation-steps 1 \
    --max-completion-length 768

## 4 — Go/no-go gate

`--lambda-think 0.0`: correctness + format, no length penalty. The question is whether GRPO
improves plain correctness over the base model at all.

Baseline to beat, on the **eval split only** (the split manifest is written into the run
directory so the baselines can be restricted to the same items):
adaptive-prompt = 87.5% accuracy at 287.5 mean tokens.

In [ ]:
!python -m train.train_grpo --output runs/gate --lambda-think 0.0 --use-vllm --max-steps 200
!cp -r runs/gate $DRIVE_RUNS/

In [ ]:
# Training traces. Watch think-rate and zero-variance, not the reward curve.
import json


def show(run="runs/gate"):
    history = json.load(open(f"{run}/log_history.json", encoding="utf-8"))
    rows = [r for r in history if "reward" in r]
    if not rows:
        print("no training rows logged")
        return

    cols = [
        ("step", "step", "{:>6}"),
        ("reward", "reward", "{:>8.3f}"),
        ("think", "rewards/metric_think_rate/mean", "{:>8.3f}"),
        ("correct", "rewards/metric_correctness/mean", "{:>8.3f}"),
        ("format", "rewards/metric_format_rate/mean", "{:>8.3f}"),
        ("thinkTok", "rewards/metric_mean_think_tokens/mean", "{:>9.1f}"),
        # Fraction of groups where every rollout scored the same -> zero
        # advantage -> zero gradient. A step like that teaches nothing.
        ("zeroStd", "frac_reward_zero_std", "{:>8.2f}"),
        # Fraction of rollouts that hit the completion cap. These never reached
        # a tool call, so they score zero correctness by construction.
        ("clipped", "completions/clipped_ratio", "{:>8.2f}"),
        ("entropy", "entropy", "{:>8.3f}"),
    ]

    print("".join(f"{name:>9s}" for name, _, _ in cols))
    for r in rows[:: max(1, len(rows) // 25)]:
        out = []
        for _, key, fmt in cols:
            v = r.get(key)
            out.append(fmt.format(v) if isinstance(v, (int, float)) else f"{'-':>8s}")
        print("".join(out))

    dead = [r for r in rows if r.get("frac_reward_zero_std", 0) == 1.0]
    print(f"\nsteps with no gradient at all: {len(dead)}/{len(rows)}")
    print("oracle think-rate is 0.174; near 0 or near 1 is collapse")


show()

## 5 — λ sweep (only after the gate passes)

Each run is independent, so a session drop costs one λ rather than the sweep. Adapters are
copied to Drive as they finish.

In [ ]:
for lam in [0.05, 0.1, 0.25, 0.5, 1.0, 2.0]:
    tag = f"lam{lam}".replace(".", "_")
    if os.path.exists(f"{DRIVE_RUNS}/{tag}/adapter_model.safetensors"):
        print(f"skipping {tag}, already done")
        continue
    print(f"\n{'='*70}\nlambda = {lam}\n{'='*70}", flush=True)
    !python -m train.train_grpo --output runs/$tag --lambda-think $lam --use-vllm --max-steps 200
    !cp -r runs/$tag $DRIVE_RUNS/

## 6 — Evaluate a trained policy

Through `generate.run_vllm --adapter`, i.e. the **same** path that produced the baselines.
Scoring a trained policy through a different pipeline than its baselines is how comparisons
quietly break. Needs vLLM, so either run this in the generation notebook's session or install
it here (see `colab/generate_baselines.ipynb` for the CUDA 13 / torch cu130 fix).

In [ ]:
# !python -m generate.run_vllm --policy adaptive --adapter runs/gate \
#     --out results/raw/vllm/qwen3-1.7b_trained_gate.jsonl

## Then, locally

Score on CPU, restricted to the eval split recorded in `runs/<name>/split_manifest.json`:

```bash
uv run python -m analysis.score_run results/raw/vllm/qwen3-1.7b_trained_gate.jsonl
```